# 04 - Diagnostics

Reads a run directory (`metrics.jsonl`, `eval_report_test.json`, the serving bundle) and shows
the learning dynamics, the direction heads' spread and bias, variance calibration (PIT,
variance vs. squared error, interval coverage) and the label balance under the deadband.

In [ ]:
# Parameters
RUN_DIR = None
RUNS_DIR = "../runs"
CSV_PATH = "../binance_btcusdt_1min_ccxt.csv"

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from scipy.stats import norm

from neural_trade.data.processor import split_arrays
from neural_trade.evaluation.frame import HORIZONS
from neural_trade.metrics.direction_labels import direction_labels_np
from neural_trade.serving.predictor import Predictor
from neural_trade.telemetry.epoch_logger import read_metrics

run_dir = Path(RUN_DIR) if RUN_DIR else max((p for p in Path(RUNS_DIR).iterdir() if (p / "metrics.jsonl").exists()), key=os.path.getmtime)
log = pd.DataFrame(read_metrics(run_dir / "metrics.jsonl"))
report = json.loads((run_dir / "eval_report_test.json").read_text()) if (run_dir / "eval_report_test.json").exists() else None
print(run_dir, "|", len(log), "epochs")

## Learning dynamics

In [ ]:
cols = [c for c in ("loss", "val_loss", "val_nll_loss", "val_crps_loss", "val_dir_mcc_h1", "val_gauss_dir_mcc_h1",
                    "val_pred_up_rate_h1", "nonfinite_grad_steps", "grad_global_norm") if c in log]
px.line(log, x="epoch", y=cols, facet_col="variable", facet_col_wrap=3, height=600).update_yaxes(matches=None).show()

## Test-block predictions

Rebuilt from the bundle and the run's config (same purged split).

In [ ]:
predictor = Predictor.from_artifacts(run_dir / "artifacts")
cfg = predictor.config.copy().override(CSV_PATH=CSV_PATH)
test = split_arrays(cfg)["test"]
frame = predictor.predict(test["X"], test["last_close"]).to_prediction_frame(
    predictor.bundle.pred_scale, predictor.bundle.pred_mean, y=test["y"], split="test")
labels = direction_labels_np(frame.y, frame.last_close, cfg.DIR_DEADBAND_BPS)
pd.DataFrame({h: {"P(up) std": frame.direction_prob[h].std(), "P(up) mean": frame.direction_prob[h].mean(),
                  "true up rate (outside deadband)": labels[h][0][labels[h][1]].mean(),
                  "share outside deadband": labels[h][1].mean(),
                  "delta std ($)": frame.delta[h].std(), "realised std ($)": frame.y[:, i].std()}
              for i, h in enumerate(HORIZONS)})

## Variance calibration

In [ ]:
pit = pd.DataFrame({h: norm.cdf((frame.y[:, i] - frame.delta[h]) / frame.sigma(h)) for i, h in enumerate(HORIZONS)})
px.histogram(pit.melt(var_name="horizon", value_name="PIT"), x="PIT", facet_col="horizon", nbins=20,
             title="PIT histogram (flat = calibrated)").show()
err2 = pd.DataFrame({"sigma_h1": frame.sigma("h1"), "abs_error_h1": np.abs(frame.y[:, 1] - frame.delta["h1"])})
err2["sigma_decile"] = pd.qcut(err2["sigma_h1"], 10, labels=False, duplicates="drop")
err2.groupby("sigma_decile")[["sigma_h1", "abs_error_h1"]].mean()

In [ ]:
pd.DataFrame({h: v["variance"] for h, v in report["model"]["horizons"].items()}) if report else "no eval report"